# 双向循环神经网络（Bidirectional RNN）

## 什么是双向RNN？

在普通的RNN中，信息只能从过去流向未来（单向）。但在很多任务中，我们需要同时利用过去和未来的信息。例如：
- **填空题**：预测句子中间的词时，既要看前文，也要看后文
- **语音识别**：识别当前音素时，后面的音素也能提供线索
- **命名实体识别**：判断一个词是否是人名时，前后文都很重要

双向RNN通过以下方式解决这个问题：
1. **前向层**：从左到右处理序列
2. **后向层**：从右到左处理序列
3. **组合输出**：将两个方向的隐状态拼接起来

## 本代码实现

本代码使用双向LSTM在时间机器数据集上进行字符级语言建模。

In [ ]:
# ==================== 导入必要的库 ====================
import torch
import RNN  # 导入自定义的RNN模块，包含数据加载和训练函数
from torch import nn
from d2l import torch as d2l

# ==================== 设置超参数 ====================
# batch_size: 每批次处理的样本数量（32个序列）
# num_steps: 每个序列的时间步数（35个字符）
# device: 使用GPU加速训练（如果可用）
batch_size, num_steps, device = 32, 35, d2l.try_gpu()

# ==================== 加载时间机器数据集 ====================
# 返回数据迭代器和词汇表
# train_iter: 用于迭代训练数据的迭代器
# vocab: 词汇表，包含所有字符到索引的映射
train_iter, vocab = RNN.load_data_time_machine(batch_size, num_steps)

# ==================== 定义模型架构 ====================
# vocab_size: 词汇表大小（输入维度）
# num_hiddens: 隐藏层单元数（256个神经元，控制模型容量）
# num_layers: LSTM层数（2层，增加模型深度）
vocab_size, num_hiddens, num_layers = len(vocab), 256, 2
num_inputs = vocab_size

# ==================== 创建双向LSTM层 ====================
# bidirectional=True: 关键参数！启用双向处理
# 这意味着：
#   - 前向层：从第1个字符到最后一个字符处理
#   - 后向层：从最后一个字符到第1个字符处理
#   - 输出维度会翻倍：每个时间步输出 2*num_hiddens 维向量
lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers, bidirectional=True)

# ==================== 封装完整模型 ====================
# RNNModel会自动处理双向LSTM的输出
# len(vocab): 输出词汇表大小，用于字符预测
model = RNN.RNNModel(lstm_layer, len(vocab))
model = model.to(device)  # 将模型移到GPU

# ==================== 训练模型 ====================
# num_epochs: 训练500个完整周期（遍历整个数据集500次）
# lr: 学习率设置为1（相对较大，因为是简单任务）
num_epochs, lr = 500, 1

# 使用自定义的训练函数进行训练
# 会输出训练过程中的困惑度（perplexity）变化
RNN.train_ch8(model, train_iter, vocab, lr, num_epochs, device)